# Clone Repository and set up the Environment

In [10]:
!pwd

/content/robust-eeg-models


In [1]:
!git clone https://github.com/VictoryChianumba/robust-eeg-models

Cloning into 'robust-eeg-models'...
remote: Enumerating objects: 547, done.
remote: Counting objects: 100% (289/289), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 547 (delta 224), reused 204 (delta 159), pack-reused 258 (from 1)
Receiving objects: 100% (547/547), 216.34 MiB | 15.08 MiB/s, done.
Resolving deltas: 100% (347/347), done.
Filtering content: 100% (2/2), 663.03 MiB | 30.85 MiB/s, done.


In [1]:
%cd robust-eeg-models

/content/robust-eeg-models


In [3]:
!git config --global user.email "chianumbav@gmial.com"
!git config --global user.name "Victory Chianumba"

In [11]:
# 1️⃣ Upgrade the package manager
!pip install --upgrade --quiet pip

!pip install torch torchvision torchaudio
!pip install mne moabb
!pip install torch-lr-finder
!pip install optuna

# Clean uninstall
!pip uninstall -y braindecode

# Install latest code from GitHub (which includes CTNet)
!pip install git+https://github.com/braindecode/braindecode.git@master --no-cache-dir

Found existing installation: braindecode 1.2.0
Uninstalling braindecode-1.2.0:
  Successfully uninstalled braindecode-1.2.0
  Cloning https://github.com/braindecode/braindecode.git (to revision master) to /tmp/pip-req-build-7kmalj96
  Running command git clone --filter=blob:none --quiet https://github.com/braindecode/braindecode.git /tmp/pip-req-build-7kmalj96
  Resolved https://github.com/braindecode/braindecode.git to commit 8305c3d04658b192f120477e90db0ff88b36177d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for braindecode: filename=braindecode-1.2.0-py3-none-any.whl size=290634 sha256=8ee065334f7899c241ddd56ec5ab7677da32e9725cefb7c6cdb7b8d881746adf
  Stored in directory: /tmp/pip-ephem-wheel-cache-f51l3l9z/wheels/b6/8b/2b/0da876924d16f36b5bdd43797902e55d90426ed71fd51642f2
Successfully built braindecode


In [4]:
import braindecode
print("Braindecode version:", braindecode.__version__)

from braindecode.models import CTNet
print("CTNet is available ✅")


Braindecode version: 1.2.0
CTNet is available ✅


In [9]:
%%writefile models/__init__.py

Overwriting models/__init__.py


In [39]:
%%writefile models/eeg_mamba_fft.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft as fft
from braindecode.models.base import EEGModuleMixin

# ------------------------------------------------------------------
# 1. FFT-Based Mamba Implementation (No external dependencies)
# ------------------------------------------------------------------
class FFTMamba(nn.Module):
    def __init__(self, d_model: int, d_state: int = 64, bidirectional: bool = True,
                 dropout: float = 0.3):
        super().__init__()
        self.d_state = d_state
        self.bidir   = bidirectional
        self.A_log   = nn.Parameter(torch.randn(d_state) * 0.02)
        self.B_proj  = nn.Linear(d_model, d_state)
        self.C_proj  = nn.Linear(d_state, d_model)
        self.D       = nn.Parameter(torch.ones(d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, _ = x.shape
        A = -torch.exp(self.A_log).unsqueeze(0)
        u = self.B_proj(x)

        n_fft = 2 ** (T - 1).bit_length()
        t = torch.arange(T, dtype=torch.float, device=x.device)
        K = torch.exp(A * t.unsqueeze(1)).to(torch.float32)
        if self.bidir:
            K = K + torch.flip(K, dims=[0])

        K_f = fft.rfft(K, n=n_fft, dim=0)
        u_f = fft.rfft(u, n=n_fft, dim=1)
        y_f = K_f.unsqueeze(0) * u_f
        y = fft.irfft(y_f, n=n_fft, dim=1)[..., :T, :]

        out = self.dropout(self.C_proj(y)) + self.D * x
        return out

class SpatialDW(nn.Module):
    """
    Spatial 1×1 conv (C → D)  followed by depth-wise 1-D temporal conv.
    Keeps (B, C, T) → (B, T, D).
    """
    def __init__(self, n_chans: int, d_model: int, kernel: int = 15, dropout: float = 0.0):
        super().__init__()
        self.spatial = nn.Conv2d(1, d_model, (n_chans, 1), bias=False)   # (B,1,C,T)→(B,D,1,T)
        self.temporal = nn.Conv1d(
            d_model, d_model, kernel_size=kernel,
            padding=kernel//2, groups=d_model, bias=False
        )
        self.cls = nn.Parameter(torch.randn(1, 1, d_model))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):               # (B, C, T)
        x = x.unsqueeze(1)              # (B,1,C,T)
        x = self.spatial(x).squeeze(2)  # (B,D,T)
        x = self.temporal(x)            # (B,D,T)
        x = x.transpose(1, 2)           # (B,T,D)
        cls = self.cls.expand(x.size(0), -1, -1)
        return torch.cat([cls, x], dim=1)   # (B,T+1,D)

class BiMambaBlock(nn.Module):
    def __init__(self, d_model, d_state=64, dropout=0.1, ffn_mult=2):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.mamba = FFTMamba(d_model, d_state=d_state, bidirectional=True, dropout=dropout)

        # depth-wise temporal conv
        self.temporal = nn.Conv1d(
            d_model, d_model, kernel_size=15, padding=7, groups=d_model, bias=False
        )

        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_mult * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_mult * d_model, d_model),
        )

    def forward(self, x):                # (B, T, D)
        x = x + self.mamba(self.norm1(x))
        # depth-wise conv
        y = self.temporal(x.transpose(1, 2)).transpose(1, 2)
        x = x + y
        x = x + self.ffn(self.norm2(x))
        return x

class TaskMoE(nn.Module):
    """Task-aware Mixture-of-Experts head."""
    def __init__(self, d_model: int, n_classes: int,
                 n_experts: int = 9, k: int = 2):
        super().__init__()
        self.experts = nn.ModuleList([
            nn.Linear(d_model, n_classes) for _ in range(n_experts)
        ])
        self.gate = nn.Linear(d_model, n_experts)
        self.k = k

    def forward(self, x):
      logits = self.gate(x)
      probs = F.softmax(logits, dim=-1)
      topk_vals, topk_idx = torch.topk(probs, self.k, dim=-1)

      # Compute all expert outputs
      expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=1)  # (B, n_experts, n_classes)

      # Select top-k experts using advanced indexing
      batch_indices = torch.arange(x.size(0), device=x.device).unsqueeze(1)  # (B, 1)
      selected_outputs = expert_outputs[batch_indices, topk_idx]  # (B, k, n_classes)

      # Weight and sum
      weights = topk_vals.unsqueeze(-1)  # (B, k, 1)
      y = (weights * selected_outputs).sum(dim=1)  # (B, n_classes)

      return y
# ------------------------------------------------------------------
# 2. Fixed Braindecode Wrapper
# ------------------------------------------------------------------
class EEGMamba(EEGModuleMixin, nn.Module):
    """Braindecode-compatible EEGMamba model."""

    def __init__(
        self,
        # Braindecode standard parameters
        n_chans=None,
        n_outputs=None,
        n_times=None,
        chs_info=None,
        input_window_seconds=None,
        sfreq=None,
        # EEGMamba specific parameters
        d_model=128,
        n_layers=8,
        d_state=64,          # bigger state
        dropout=0.3,         # overall dropout
        n_experts=9,
        k=2,
        # Backward compatibility aliases
        in_chans=None,
        n_classes=None,
        input_window_samples=None,
    ):

        # Initialize base class
        super().__init__(
            n_outputs=n_outputs,
            n_chans=n_chans,
            chs_info=chs_info,
            n_times=n_times,
            input_window_seconds=input_window_seconds,
            sfreq=sfreq,
        )

        # Store model parameters
        self.d_model = d_model
        self.n_layers = n_layers
        self.n_experts = n_experts
        self.k = k
        self.d_state = d_state
        self.dropout = dropout


        # Build the model
        self.st_dw = SpatialDW(self.n_chans, d_model)
        self.layers = nn.ModuleList([
            BiMambaBlock(d_model, d_state=d_state, dropout=dropout) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

        # Simple classification head (no MoE for standard compatibility)
        self.classifier = nn.Linear(d_model, self.n_outputs)

        # Optional: MoE head (use when you want advanced features)
        self.moe_head = TaskMoE(d_model, self.n_outputs, n_experts, k)
        self.use_moe = False  # Toggle this for MoE vs standard mode

        # Initialize weights
        self._initialize_weights()

    def forward(self, x):
        """
        Forward pass compatible with Braindecode.

        Args:
            x: Input tensor [batch_size, n_chans, n_times]

        Returns:
            logits: Output tensor [batch_size, n_outputs]
        """
        # Spatial-temporal adaptive processing
        x = self.st_dw(x)  # (B, T+1, D)

        # Bidirectional Mamba layers
        for layer in self.layers:
            x = layer(x)

        # Use class token
        x = self.norm(x).mean(dim=1)   # (B, D)
        # x = self.norm(x[:, 0])

        # Classification
        if self.use_moe:
            return self.moe_head(x)  # Only expects tensor
        else:
            return self.classifier(x)  # Only returns tensor

    def get_output_shape(self):
        """Required method for Braindecode compatibility."""
        with torch.no_grad():
            dummy_input = torch.zeros(1, self.n_chans, self.n_times)
            output = self.forward(dummy_input)
            return output.shape

    def enable_moe(self, enable=True):
        """Enable/disable MoE head."""
        self.use_moe = enable

    def _initialize_weights(self):
        """Initialize model weights."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LayerNorm):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

# ------------------------------------------------------------------
# 3. Factory Function for Easy Usage
# ------------------------------------------------------------------
def create_eegmamba(n_chans, n_outputs, n_times, **kwargs):
    """
    Factory function to create EEGMamba model.

    Args:
        n_chans: Number of EEG channels
        n_outputs: Number of classes
        n_times: Number of time samples
        **kwargs: Additional model parameters

    Returns:
        EEGMamba model instance
    """
    return EEGMamba(
        n_chans=n_chans,
        n_outputs=n_outputs,
        n_times=n_times,
        **kwargs
    )


Overwriting models/eeg_mamba_fft.py


In [40]:
import importlib
import models.eeg_mamba_fft
import models.__init__
importlib.reload(models.eeg_mamba_fft)
importlib.reload(models.__init__)

from models.eeg_mamba_fft import create_eegmamba, EEGMamba

In [46]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn
import torch.optim as optim

import importlib

import numpy as np
import os
import sys
import pickle
import json
import random
import time
import datetime

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import defaultdict


In [82]:
from braindecode import EEGClassifier
from braindecode.models import EEGNetv4, Deep4Net, CTNet
from braindecode.datasets import MOABBDataset
from braindecode.augmentation import FrequencyShift, GaussianNoise, AugmentedDataLoader, Compose, SmoothTimeMask, Mixup
from braindecode.training import mixup_criterion

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from skorch.helper import predefined_split
from skorch.callbacks import LRScheduler, EarlyStopping, Checkpoint

from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR, CosineAnnealingWarmRestarts

from models.eeg_mamba_fft import create_eegmamba, EEGMamba

# Loading data for training

In [77]:
import numpy as np
from braindecode.datasets import MOABBDataset
from braindecode.preprocessing import Preprocessor, exponential_moving_standardize, preprocess
from braindecode.preprocessing import create_windows_from_events
from sklearn.model_selection import train_test_split
from skorch.helper import SliceDataset
from torch.utils.data import Subset

dataset = MOABBDataset(
    dataset_name='BNCI2014001', subject_ids=[1]
)

#----------------------------------------------------------------------
# After loading we preprocess

low_cut_hz = 4.0  # low cut frequency for filtering
high_cut_hz = 38.0  # high cut frequency for filtering
# Parameters for exponential moving standardization
factor_new = 1e-3
init_block_size = 1000

preprocessors = [
    Preprocessor("pick_types", eeg=True, meg=False, stim=False),  # Keep EEG sensors
    Preprocessor(
        lambda data, factor: np.multiply(data, factor),  # Convert from V to uV
        factor=1e6,
    ),
    Preprocessor("filter", l_freq=low_cut_hz, h_freq=high_cut_hz),  # Bandpass filter
    Preprocessor(
        exponential_moving_standardize,  # Exponential moving standardization
        factor_new=factor_new,
        init_block_size=init_block_size,
    ),
]

# Preprocess the data
preprocess(dataset, preprocessors, n_jobs=-1)

#-----------------------------------------------------------------------

trial_start_offset_seconds = -0.5
# Extract sampling frequency, check that they are same in all datasets
sfreq = dataset.datasets[0].raw.info["sfreq"]
assert all([ds.raw.info["sfreq"] == sfreq for ds in dataset.datasets])
# Calculate the window start offset in samples.
trial_start_offset_samples = int(trial_start_offset_seconds * sfreq)

# Create windows using braindecode function for this. It needs parameters to
# define how windows should be used.
windows_dataset = create_windows_from_events(
    dataset,
    trial_start_offset_samples = int(-0.5 * sfreq) , # -0.5s before cue
    trial_stop_offset_samples = 0,   # 4.0s after cue (t=2s to t=6s)
    preload=True,
    # verbose=0
)

sample_window = windows_dataset[0][0]
print(f"Actual window shape: {sample_window.shape}")
print(f"Expected for EEGNet: (n_channels, ~1000) for 4s at 250Hz")

# Check the window timing metadata
metadata = windows_dataset.get_metadata()
print("Start samples:", metadata['i_start_in_trial'].unique())
print("Stop samples:", metadata['i_stop_in_trial'].unique())
print("Window sizes:", (metadata['i_stop_in_trial'] - metadata['i_start_in_trial']).unique())

/usr/local/lib/python3.11/dist-packages/braindecode/preprocessing/preprocess.py:71: UserWarning: Preprocessing choices with lambda functions cannot be saved.
  warn("Preprocessing choices with lambda functions cannot be saved.")


Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Actual window shape: (22, 1125)
Expected for EEGNet: (n_channels, ~1000) for

In [80]:
def load_subject_data_cached(dataset, subject_id):
    cache_file = f'cache/subject_{subject_id}_processed.pkl'

    if os.path.exists(cache_file):
        # Load from cache - instant!
        with open(cache_file, 'rb') as f:
            return pickle.load(f)

    # Process and cache
    train_set, test_set, train_subset, val_subset = load_subject_data(dataset,subject_id)

    os.makedirs('cache', exist_ok=True)
    with open(cache_file, 'wb') as f:
        pickle.dump((train_set, test_set, train_subset, val_subset), f)


    return train_set, test_set, train_subset, val_subset

def load_subject_data(dataset, subject_id):

    import numpy as np
    from braindecode.datasets import MOABBDataset
    from braindecode.preprocessing import Preprocessor, exponential_moving_standardize, preprocess
    from braindecode.preprocessing import create_windows_from_events
    from sklearn.model_selection import train_test_split
    from skorch.helper import SliceDataset
    from torch.utils.data import Subset

    dataset = MOABBDataset(
        dataset_name=dataset, subject_ids=[subject_id]
    )

    #----------------------------------------------------------------------
    # After loading we preprocess

    low_cut_hz = 4.0  # low cut frequency for filtering
    high_cut_hz = 38.0  # high cut frequency for filtering
    # Parameters for exponential moving standardization
    factor_new = 1e-3
    init_block_size = 750

    preprocessors = [
        Preprocessor("pick_types", eeg=True, meg=False, stim=False),  # Keep EEG sensors
        Preprocessor(
            lambda data, factor: np.multiply(data, factor),  # Convert from V to uV
            factor=1e6,
        ),
        Preprocessor("filter", l_freq=low_cut_hz, h_freq=high_cut_hz),  # Bandpass filter
        Preprocessor(
            exponential_moving_standardize,  # Exponential moving standardization
            factor_new=factor_new,
            init_block_size=init_block_size,
        ),
    ]

    # Preprocess the data
    preprocess(dataset, preprocessors, n_jobs=-1)

    #-----------------------------------------------------------------------

    trial_start_offset_seconds = -0.5
    # Extract sampling frequency, check that they are same in all datasets
    sfreq = dataset.datasets[0].raw.info["sfreq"]
    assert all([ds.raw.info["sfreq"] == sfreq for ds in dataset.datasets])
    # Calculate the window start offset in samples.
    trial_start_offset_samples = int(trial_start_offset_seconds * sfreq)

    # Create windows using braindecode function for this. It needs parameters to
    # define how windows should be used.
    windows_dataset = create_windows_from_events(
        dataset,
        trial_start_offset_samples=trial_start_offset_samples,
        trial_stop_offset_samples=0,
        preload=True,
        # verbose=0
    )

    # ----------------------------------------------------------------------
    # Split into train and test
    splitted = windows_dataset.split("session")
    train_set = splitted["0train"]  # Session train
    test_set = splitted["1test"]  # Session evaluation

    # ----------------------------------------------------------------------
    # Split into train, val subsets

    X_train = SliceDataset(train_set, idx=0)
    y_train = np.array([y for y in SliceDataset(train_set, idx=1)])
    train_indices, val_indices = train_test_split(
        X_train.indices_, test_size=0.2, shuffle=False
    )
    train_subset = Subset(train_set, train_indices)
    val_subset = Subset(train_set, val_indices)

    return train_set, test_set, train_subset, val_subset


## Data loading sanity check

In [14]:
# Run once or sanity check
subject_id = 2
train_set, test_set, train_subset, val_subset = load_subject_data_cached(subject_id)

/usr/local/lib/python3.11/dist-packages/moabb/datasets/download.py:56: RuntimeWarning: Setting non-standard config type: "MNE_DATASETS_BNCI_PATH"
  set_config(key, get_config("MNE_DATA"))


MNE_DATA is not already configured. It will be set to default location in the home directory - /root/mne_data
All datasets will be downloaded to this location, if anything is already downloaded, please move manually to this location


/usr/local/lib/python3.11/dist-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'lampx.tugraz.at'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
100%|█████████████████████████████████████| 43.1M/43.1M [00:00<00:00, 77.8GB/s]
SHA256 hash of downloaded file: 5ddd5cb520b1692c3ba1363f48d98f58f0e46f3699ee50d749947950fc39db27
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.
/usr/local/lib/python3.11/dist-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'lampx.tugraz.at'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
100%|█████████████████████████████████████| 44.2M/44.2M [

Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']
Used Annotations descriptions: ['feet', 'left_hand', 'right_hand', 'tongue']


# Load and inspect model

In [26]:

from braindecode.util import set_random_seeds

cuda = torch.cuda.is_available()  # check if GPU is available, if True chooses to use it
device = "cuda" if cuda else "cpu"
if cuda:
    torch.backends.cudnn.benchmark = True
seed = 20200220
set_random_seeds(seed=seed, cuda=cuda)

# Extract number of chans and time steps from dataset
n_channels = train_subset[0][0].shape[0]
n_times = train_subset[0][0].shape[1]
n_classes = len(np.unique([train_subset[i][1] for i in range(len(train_subset))]))
classes = list(range(n_classes))

model = EEGMamba(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,
    n_layers = 2,
    n_experts = 8

)

model.enable_moe(True)

# Display torchinfo table describing the model
print(model)

# Send model to GPU
if cuda:
    model.cuda()

/usr/local/lib/python3.11/dist-packages/braindecode/util.py:52: UserWarning: torch.backends.cudnn.benchmark was set to True which may results in lack of reproducibility. In some cases to ensure reproducibility you may need to set torch.backends.cudnn.benchmark to False.
  warn(


Layer (type (var_name):depth-idx)             Input Shape               Output Shape              Param #                   Kernel Shape
EEGMamba (EEGMamba)                           [1, 22, 1125]             [1, 4]                    516                       --
├─STAdaptive (st_adaptive): 1-1               [1, 22, 1125]             [1, 1126, 128]            128                       --
│    └─Conv1d (proj): 2-1                     [1, 22, 1125]             [1, 128, 1125]            2,816                     [1]
├─ModuleList (layers): 1-2                    --                        --                        --                        --
│    └─BiMambaBlock (0): 2-2                  [1, 1126, 128]            [1, 1126, 128]            --                        --
│    │    └─LayerNorm (norm1): 3-1            [1, 1126, 128]            [1, 1126, 128]            256                       --
│    │    └─FFTMamba (mamba): 3-2             [1, 1126, 128]            [1, 1126, 128]            4,

# Training

# Set model hyper params

In [24]:

eegnet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam}

deepconvnet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam}

CTNet_params = {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.AdamW}
alt_params = {'lr': 1e-3, 'batch_size': 128, 'weight_decay': 5e-3, 'optimizer': torch.optim.AdamW}

mamba_params = {'lr': 2e-4, 'batch_size': 128, 'weight_decay': 1e-6, 'optimizer': torch.optim.AdamW}
mamba_params1 = {'lr': 0.0016, 'batch_size': 128, 'weight_decay': 5e-4, 'optimizer': torch.optim.AdamW}

n_epochs = 500

In [63]:
!rm -rf results/

In [ ]:
def make_scheduler(optimizer, last_epoch=-1):
    return CosineAnnealingWarmRestarts(optimizer, T_0=300, T_mult=1, eta_min=1e-6, last_epoch=last_epoch)

# ==============================================================================

def train_single_run(model_name, subject_id, seed, dataset):

    # 0. RNG reproducibility ---------------------------------------------------

    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    rng_state_np   = np.random.get_state()
    rng_state_torch = torch.get_rng_state()

    print(f"\n=== Processing Subject {subject_id} for seed {seed} ===")

    # 1. data ------------------------------------------------------------------
    # Load data
    train_set, test_set, train_subset, val_subset = load_subject_data_cached(dataset, subject_id)

    # expose indices explicitly
    train_idx = np.arange(len(train_subset))
    val_idx   = np.arange(len(val_subset)) + len(train_subset)
    test_idx  = np.arange(len(test_set))

    # 2. model & config --------------------------------------------------------

    # Get model config
    config = MODEL_CONFIGS[model_name]

    # Extract model params from dataset, initialise model and set hyper-parameters
    n_classes = len(torch.unique(torch.tensor([sample[1] for sample in train_subset])))
    n_channels = train_subset[0][0].shape[0]
    n_times = train_subset[0][0].shape[1]

    model = config['model_class'](
        n_chans=n_channels,
        n_outputs=n_classes,
        n_times=n_times,
    )

    # Special handling for EEGMamba
    if model_name == 'EEGMamba':
        model.enable_moe(False)  # Use standard classifier for baseline. Paper explicitly removes moe modules for
                                # single use mamba
        # Enable MoE head instead of standard classifier (For multi-use only)
        model.use_moe = False

    # 3. callbacks --------------------------------------------------------------
    if config['training']['scheduler']:
        callbacks = ["accuracy", ("lr_scheduler", LRScheduler(make_scheduler))]
    else:
        callbacks = ["accuracy"]

    # 4. fit -------------------------------------------------------------------

    # Create new classifier with best parameters
    clf = EEGClassifier(
        model,
        criterion=torch.nn.CrossEntropyLoss,
        train_split=predefined_split(val_subset),  # Use all training data
        optimizer=config['training']['optimizer'],
        optimizer__lr=config['training']['lr'],
        optimizer__weight_decay=config['training']['weight_decay'],
        batch_size=config['training']['batch_size'],
        callbacks=callbacks,
        device=device,
        classes=classes,
        max_epochs=500,
    )

    # Train on full training set
    clf.fit(train_subset, y=None)

    # 5. test accuracy ---------------------------------------------------------

    # Evaluate the model after training
    y_test = test_set.get_metadata().target
    test_accuracy = clf.score(test_set, y = y_test)

    # 6. save everything -------------------------------------------------------

    _save_run(model_name, subject_id, seed,
              clf, test_set, rng_state_np, rng_state_torch,
              train_idx, val_idx, test_idx)

    return test_accuracy

# ==============================================================================

# Generate final baseline table
def create_baseline_table(results):
    """Create a nice table of baselines"""
    rows = []

    for model_name in results.keys():
        for subject_id in subjects:
            scores = results[model_name][subject_id]
            valid_scores = [s for s in scores if not np.isnan(s)]

            if valid_scores:
                mean_acc = np.mean(valid_scores)
                std_acc = np.std(valid_scores)
                n_valid = len(valid_scores)
            else:
                mean_acc = std_acc = n_valid = np.nan

            rows.append({
                'Model': model_name,
                'Subject': subject_id,
                'Mean_Accuracy': mean_acc,
                'Std_Accuracy': std_acc,
                'N_Valid_Runs': n_valid,
                'Individual_Scores': scores
            })

    return pd.DataFrame(rows)


# ==============================================================================

def _save_run(model_name, subject_id, seed, clf, test_set, rng_state_np, rng_state_torch, train_idx, val_idx, test_idx):
    """
    clf          : fitted skorch net
    test_set     : braindecode Dataset
    *_idx        : np.ndarray of ints
    """
    base = f"{SAVE_DIR}/{model_name}_S{subject_id}_seed{seed}"
    os.makedirs(base, exist_ok=True)

    # Save checkpoint
    torch.save(clf.module_.state_dict(), f"{base}/checkpoint.pth")

    # Save training curves
    hist = clf.history_
    train_acc = [e["train_accuracy"] for e in hist]  # Updated key
    val_acc = [e["valid_accuracy"] for e in hist]    # Updated key
    train_loss = [e["train_loss"] for e in hist]
    val_loss = [e["valid_loss"] for e in hist]
    curves = {"train_acc": train_acc, "val_acc": val_acc, "train_loss": train_loss, "val_loss": val_loss}
    json.dump(curves, open(f"{base}/curves.json", "w"), indent=2)

    # Save test loss vector
    X_test = np.stack([test_set[i][0] for i in range(len(test_set))])
    y_test = np.array(test_set.get_metadata().target)
    clf.module_.eval()
    with torch.no_grad():
        logits = clf.infer(X_test)
    loss_fn = torch.nn.CrossEntropyLoss(reduction="none")
    y_test_tensor = torch.tensor(y_test).to(logits.device)
    loss_vec = loss_fn(logits.to(device), y_test_tensor.to(device)).cpu().numpy()
    np.save(f"{base}/test_loss_vector.npy", loss_vec)

    # Save RNG states
    pickle.dump({"numpy": rng_state_np, "torch": rng_state_torch}, open(f"{base}/rng_state.pkl", "wb"))

    # Save split indices
    splits = {"train_idx": train_idx.tolist(), "val_idx": val_idx.tolist(), "test_idx": test_idx.tolist()}
    json.dump(splits, open(f"{base}/splits.json", "w"), indent=2)
# ----------------------------------------------------------------------------------------------------------------
# Baseline Run
# ----------------------------------------------------------------------------------------------------------------

seeds = [42, 123, 2024, 31415, 999]   # any 3–5 different seeds

datasets = {
    "BNCIv2": ("BNCI2014001", 9),
    "DEAP": 32
}

dataset, n_subjects = datasets["BNCIv2"]
subjects = list(range(1, n_subjects+1))


SAVE_DIR = "results"          # change if you want
os.makedirs(SAVE_DIR, exist_ok=True)

# Define your models and their hyperparameters
MODEL_CONFIGS = {
    # 'EEGNet': {
    #     'model_class': EEGNetv4,
    #     'training': {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam, 'scheduler': False}
    # },
    # 'DeepConvNet': {
    #     'model_class': Deep4Net,
    #     'training': {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam, 'scheduler': False}
    # },
    # 'CTNet': {
    #     'model_class': CTNet,
    #     'training':  {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.AdamW, 'scheduler': True}
    # },
    'EEGMamba': {
        'model_class': EEGMamba,
        'training': {'lr': 2e-4, 'batch_size': 128, 'weight_decay': 1e-6, 'optimizer': torch.optim.AdamW, 'scheduler': True}
    }
}

# Store all results: results[model_name][subject_id] = [acc1, acc2, acc3, acc4, acc5]
all_results = defaultdict(lambda: defaultdict(list))

# Main loop
for model_name in MODEL_CONFIGS.keys():
    print(f"\n{'='*60}")
    print(f"RUNNING BASELINE FOR {model_name.upper()}")
    print(f"{'='*60}")

    for subject_id in subjects:
        print(f"\n--- Subject {subject_id} ---")

        subject_scores = []
        for seed in seeds:
            try:
                accuracy = train_single_run(model_name, subject_id, seed, dataset)
                subject_scores.append(accuracy)
                print(f"  Seed {seed}: {accuracy:.4f}")
            except Exception as e:
                print(f"  Seed {seed}: FAILED ({e})")
                subject_scores.append(np.nan)

        # Store results for this (model, subject) pair
        all_results[model_name][subject_id] = subject_scores

        # Calculate stats for this subject
        valid_scores = [s for s in subject_scores if not np.isnan(s)]
        if valid_scores:
            mean_acc = np.mean(valid_scores)
            std_acc = np.std(valid_scores)
            print(f"  Subject {subject_id} baseline: {mean_acc:.4f} ± {std_acc:.4f}")
        else:
            print(f"  Subject {subject_id}: ALL RUNS FAILED")

# Create and display results
baseline_df = create_baseline_table(all_results)
print(f"\n{'='*80}")
print("FINAL BASELINE RESULTS")
print(f"{'='*80}")

# Subject-wise baselines
for model_name in MODEL_CONFIGS.keys():
    print(f"\n{model_name}:")
    model_data = baseline_df[baseline_df['Model'] == model_name]

    subject_means = []
    for _, row in model_data.iterrows():
        if not np.isnan(row['Mean_Accuracy']):
            print(f"  Subject {row['Subject']}: {row['Mean_Accuracy']:.4f} ± {row['Std_Accuracy']:.4f}")
            subject_means.append(row['Mean_Accuracy'])
        else:
            print(f"  Subject {row['Subject']}: FAILED")

    # Dataset-wide average
    if subject_means:
        dataset_mean = np.mean(subject_means)
        dataset_std = np.std(subject_means)
        print(f"  → Dataset average: {dataset_mean:.4f} ± {dataset_std:.4f}")
    else:
        print(f"  → Dataset average: FAILED")

# Save results
baseline_df.to_csv('baseline_results.csv', index=False)
print(f"\nResults saved to baseline_results.csv")


RUNNING BASELINE FOR EEGMAMBA

--- Subject 1 ---

=== Processing Subject 1 for seed 42 ===
  epoch    train_accuracy    train_loss    valid_acc    valid_accuracy    valid_loss      lr     dur
-------  ----------------  ------------  -----------  ----------------  ------------  ------  ------
      1            0.2478        1.4067       0.2069            0.2069        1.3998  0.0002  0.5241
      2            0.2652        1.3942       0.2414            0.2414        1.3851  0.0002  0.5277
      3            0.2565        1.3910       0.3448            0.3448        1.3811  0.0002  0.5276
      4            0.2348        1.3955       0.3103            0.3103        1.3783  0.0002  0.5258
      5            0.2391        1.4069       0.3103            0.3103        1.3739  0.0002  0.5265
      6            0.2348        1.3901       0.3103            0.3103        1.3720  0.0002  0.5239
      7            0.2391        1.3897       0.3103            0.3103        1.3722  0.0002  0.5249

In [67]:
all_results

defaultdict(<function __main__.<lambda>()>,
            {'EEGNet': defaultdict(list,
                         {1: [0.6388888888888888,
                           0.6770833333333334,
                           0.6423611111111112,
                           0.6076388888888888,
                           0.625],
                          2: [0.4027777777777778,
                           0.3888888888888889,
                           0.4236111111111111,
                           0.5034722222222222,
                           0.4826388888888889],
                          3: [0.8194444444444444,
                           0.7638888888888888,
                           0.7743055555555556,
                           0.7395833333333334,
                           0.78125],
                          4: [0.4652777777777778,
                           0.4861111111111111,
                           0.4791666666666667,
                           0.4895833333333333,
                           0.48

In [69]:
def make_scheduler(optimizer, last_epoch=-1):
    return CosineAnnealingWarmRestarts(optimizer, T_0=300, T_mult=1, eta_min=1e-6, last_epoch=last_epoch)

SAVE_DIR = "results"          # change if you want
os.makedirs(SAVE_DIR, exist_ok=True)

# Load tehe training data
train_set, test_set, train_subset, val_subset= load_subject_data_cached("BNCI2014001", 4)

# Build transforms list
transforms = [
    FrequencyShift(probability=0.3, sfreq=250, max_delta_freq=0.3),
    GaussianNoise(probability=0.3, std=0.0),
    # Mixup(alpha=0.2,  beta_per_sample=True),               # ← returns (x, (y1, y2, lam))
]


# Extract model params from dataset, initialise model and set hyper-parameters
n_classes = len(torch.unique(torch.tensor([sample[1] for sample in train_subset])))
classes = list(range(n_classes))
n_channels = train_subset[0][0].shape[0]
n_times = train_subset[0][0].shape[1]

model = EEGNetv4(
    n_chans=n_channels,
    n_outputs=n_classes,
    n_times=n_times,


)

# Hyper params
params = eegnet_params

# Toggle this to enable mamba (For EEGMamba only)
# model.enable_moe(True)

# Enable MoE head instead of standard classifier (For EEGMamba only)
# model.use_moe = False

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

def make_scheduler(optimizer, last_epoch=-1):
    warmup = LinearLR(optimizer, start_factor=0.1, total_iters=10, last_epoch=last_epoch)
    cosine = CosineAnnealingLR(optimizer, T_max=n_epochs-10, last_epoch=last_epoch)
    return CosineAnnealingWarmRestarts(optimizer, T_0=30, T_mult=1)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=30, T_mult=1)

# Create new classifier with best parameters
clf = EEGClassifier(
    model,
    # iterator_train=AugmentedDataLoader,
    # iterator_train__transforms=transforms,
    # iterator_train__shuffle=True,

    # dataset = aug_train,
    criterion=torch.nn.CrossEntropyLoss,
    # criterion__base_criterion=torch.nn.CrossEntropyLoss(reduction='none'),

    train_split=predefined_split(val_subset),  # Use all training data

    optimizer=params['optimizer'],

    # Looking at subjects
    # optimizer = torch.optim.SGD,
    # optimizer__momentum=0.9,

    optimizer__lr=params['lr'],
    optimizer__weight_decay=params['weight_decay'],
    batch_size=params['batch_size'],
    callbacks=[
        "accuracy",
        # ("lr_scheduler", LRScheduler(make_scheduler))
        # ("lr_scheduler", LRScheduler("CosineAnnealingLR", T_max=n_epochs - 1)),
        # ("early_stopping", EarlyStopping(patience=200, monitor="valid_acc")),
    ],
    device=device,
    classes=classes,
    max_epochs=500,
)

# Train on full training set
clf.fit(train_subset, y=None)

# Evaluate the model after training
y_test = test_set.get_metadata().target
test_acc = clf.score(test_set, y=y_test)

print(f"Val acc with MoE: {(test_acc * 100):.2f}%")

  epoch    train_accuracy    train_loss    valid_acc    valid_accuracy    valid_loss     dur
-------  ----------------  ------------  -----------  ----------------  ------------  ------
      1            0.2652        1.4163       0.3103            0.3103        1.3862  0.0616
      2            0.3478        1.3977       0.2586            0.2586        1.3857  0.0579
      3            0.3522        1.3642       0.2586            0.2586        1.3852  0.0587
      4            0.3696        1.3437       0.2241            0.2241        1.3845  0.0587
      5            0.3957        1.3392       0.2586            0.2586        1.3836  0.0579
      6            0.4217        1.3110       0.3103            0.3103        1.3826  0.0571
      7            0.4348        1.3033       0.3621            0.3621        1.3813  0.0574
      8            0.4478        1.2877       0.3276            0.3276        1.3798  0.0571
      9            0.4565        1.2586       0.3103            0.3103

In [70]:
import numpy as np
print(np.bincount(y_test))

[72 72 72 72]


In [90]:
def make_scheduler(optimizer, last_epoch=-1):
    return CosineAnnealingWarmRestarts(optimizer, T_0=300, T_mult=1, eta_min=1e-6, last_epoch=last_epoch)

# ==============================================================================

def train_single_run(model_name, subject_id, seed, dataset):

    # 0. RNG reproducibility ---------------------------------------------------

    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    rng_state_np   = np.random.get_state()
    rng_state_torch = torch.get_rng_state()

    print(f"\n=== Processing Subject {subject_id} for seed {seed} ===")

    # 1. data ------------------------------------------------------------------
    # Load data
    train_set, test_set, train_subset, val_subset = load_subject_data_cached(dataset, subject_id)

    # expose indices explicitly
    train_idx = np.arange(len(train_subset))
    val_idx   = np.arange(len(val_subset)) + len(train_subset)
    test_idx  = np.arange(len(test_set))

    # 2. model & config --------------------------------------------------------

    # Get model config
    config = MODEL_CONFIGS[model_name]

    # Extract model params from dataset, initialise model and set hyper-parameters
    n_classes = len(torch.unique(torch.tensor([sample[1] for sample in train_subset])))
    n_channels = train_subset[0][0].shape[0]
    n_times = train_subset[0][0].shape[1]

    model = config['model_class'](
        n_chans=n_channels,
        n_outputs=n_classes,
        n_times=n_times,
    )

    # Special handling for EEGMamba
    if model_name == 'EEGMamba':
        model.enable_moe(False)  # Use standard classifier for baseline. Paper explicitly removes moe modules for
                                # single use mamba
        # Enable MoE head instead of standard classifier (For multi-use only)
        model.use_moe = False

    # 3. callbacks --------------------------------------------------------------
    if config['training']['scheduler']:
        callbacks = ["accuracy", ("lr_scheduler", LRScheduler(make_scheduler))]
    else:
        callbacks = ["accuracy", ]

    # 4. fit -------------------------------------------------------------------

    # Create new classifier with best parameters
    clf = EEGClassifier(
        model,
        criterion=torch.nn.CrossEntropyLoss,
        train_split=predefined_split(val_subset),  # Use all training data
        optimizer=config['training']['optimizer'],
        optimizer__lr=config['training']['lr'],
        optimizer__weight_decay=config['training']['weight_decay'],
        batch_size=config['training']['batch_size'],
        callbacks=callbacks,
        device=device,
        classes=classes,
        max_epochs=500,
    )

    # Train on full training set
    clf.fit(train_subset, y=None)

    # 5. test accuracy ---------------------------------------------------------

    # Evaluate the model after training
    y_test = test_set.get_metadata().target
    test_accuracy = clf.score(test_set, y = y_test)

    # 6. save everything -------------------------------------------------------

    _save_run(model_name, subject_id, seed,
              clf, test_set, rng_state_np, rng_state_torch,
              train_idx, val_idx, test_idx)

    return test_accuracy


seeds = [42, 123, 2024, 31415, 999]   # any 3–5 different seeds

datasets = {
    "BNCIv2": ("BNCI2014001", 9),
    "DEAP": 32
}

dataset, n_subjects = datasets["BNCIv2"]
subjects = list(range(1, n_subjects+1))


SAVE_DIR = "results"          # change if you want
os.makedirs(SAVE_DIR, exist_ok=True)

# Define your models and their hyperparameters
MODEL_CONFIGS = {
    'EEGNet': {
        'model_class': EEGNetv4,
        'training': {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam, 'scheduler': False}
    },
    'DeepConvNet': {
        'model_class': Deep4Net,
        'training': {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.Adam, 'scheduler': False}
    },
    'CTNet': {
        'model_class': CTNet,
        'training':  {'lr': 1e-3, 'batch_size': 64, 'weight_decay': 1e-4, 'optimizer': torch.optim.AdamW, 'scheduler': True}
    },
    'EEGMamba': {
        'model_class': EEGMamba,
        'training': {'lr': 2e-4, 'batch_size': 128, 'weight_decay': 1e-6, 'optimizer': torch.optim.AdamW, 'scheduler': True}
    }
}


accuracy = train_single_run('EEGNet', 1, 42, dataset)
print(accuracy)


=== Processing Subject 1 for seed 42 ===
  epoch    train_accuracy    train_loss    valid_acc    valid_accuracy    valid_loss     dur
-------  ----------------  ------------  -----------  ----------------  ------------  ------
      1            0.2783        1.4082       0.2759            0.2759        1.3860  0.0602
      2            0.2870        1.3743       0.3621            0.3621        1.3858  0.0576
      3            0.3652        1.3585       0.3103            0.3103        1.3858  0.0580
      4            0.3826        1.3277       0.2586            0.2586        1.3858  0.0572
      5            0.4174        1.3192       0.2759            0.2759        1.3857  0.0583
      6            0.4130        1.2936       0.3103            0.3103        1.3857  0.0603
      7            0.4391        1.2693       0.2586            0.2586        1.3856  0.0592
      8            0.4261        1.2610       0.2241            0.2241        1.3855  0.0585
      9            0.4261   

In [59]:
def _save_run(model_name, subject_id, seed, clf, test_set, rng_state_np, rng_state_torch, train_idx, val_idx, test_idx):
    base = f"{SAVE_DIR}/{model_name}_S{subject_id}_seed{seed}"
    os.makedirs(base, exist_ok=True)

    # Save checkpoint
    torch.save(clf.module_.state_dict(), f"{base}/checkpoint.pth")

    # Save training curves
    hist = clf.history_
    train_acc = [e["train_accuracy"] for e in hist]  # Updated key
    val_acc = [e["valid_accuracy"] for e in hist]    # Updated key
    train_loss = [e["train_loss"] for e in hist]
    val_loss = [e["valid_loss"] for e in hist]
    curves = {"train_acc": train_acc, "val_acc": val_acc, "train_loss": train_loss, "val_loss": val_loss}
    json.dump(curves, open(f"{base}/curves.json", "w"), indent=2)

    # Save test loss vector
    X_test = np.stack([test_set[i][0] for i in range(len(test_set))])
    y_test = np.array(test_set.get_metadata().target)
    clf.module_.eval()
    with torch.no_grad():
        logits = clf.infer(X_test)
    loss_fn = torch.nn.CrossEntropyLoss(reduction="none")
    y_test_tensor = torch.tensor(y_test).to(logits.device)
    loss_vec = loss_fn(logits.to(device), y_test_tensor.to(device)).cpu().numpy()
    np.save(f"{base}/test_loss_vector.npy", loss_vec)

    # Save RNG states
    pickle.dump({"numpy": rng_state_np, "torch": rng_state_torch}, open(f"{base}/rng_state.pkl", "wb"))

    # Save split indices
    splits = {"train_idx": train_idx.tolist(), "val_idx": val_idx.tolist(), "test_idx": test_idx.tolist()}
    json.dump(splits, open(f"{base}/splits.json", "w"), indent=2)